## Preprocessing the Dataset
UltraFeedback Binarized — Final Data Prep
SFT: ~6k | RM: ~8-9k | PPO prompts: ~1k | Test: 500
Model: Qwen2.5-1.5B

## 1- Importing the libraries

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer
import json, random
from pathlib import Path

## 2- Setting the Configs

In [2]:
MODEL_NAME    = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_LENGTH    = 512

SFT_SAMPLES   = 6000
RM_SAMPLES    = 8500   
PPO_SAMPLES   = 1000
TEST_SAMPLES  = 500

SEED          = 42
OUTPUT_DIR    = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)

## 3- Tokenizer loading

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 4 - Load the Dataset

In [4]:
print("Loading ultrafeedback_binarized...")
ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")
print(f"Total raw samples: {len(ds)}")

Loading ultrafeedback_binarized...


README.md: 0.00B [00:00, ?B/s]

data/train_prefs-00000-of-00001.parquet:   0%|          | 0.00/226M [00:00<?, ?B/s]

data/test_prefs-00000-of-00001.parquet:   0%|          | 0.00/7.29M [00:00<?, ?B/s]

data/test_sft-00000-of-00001.parquet:   0%|          | 0.00/3.72M [00:00<?, ?B/s]

...

Generating test_prefs split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/61135 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Total raw samples: 61135


## Helpers

In [5]:
def get_prompt(sample):
    for msg in sample["chosen"]:
        if msg["role"] == "user":
            return msg["content"].strip()
    return None

def get_response(messages):
    for msg in messages:
        if msg["role"] == "assistant":
            return msg["content"].strip()
    return None

## Quality Filter
drop samples where chosen/rejected are too similar - these are the noisy ones

In [6]:
def is_clean(sample):
    chosen_resp   = get_response(sample["chosen"])
    rejected_resp = get_response(sample["rejected"])

    if not chosen_resp or not rejected_resp:
        return False

    # too short
    if len(chosen_resp) < 50 or len(rejected_resp) < 50:
        return False

    # too long — will get truncated and lose meaning
    if len(chosen_resp) > 3000 or len(rejected_resp) > 3000:
        return False

    # nearly identical length = marginal quality diff = noisy label
    len_ratio = len(chosen_resp) / (len(rejected_resp) + 1)
    if 0.95 < len_ratio < 1.05:
        return False

    # use GPT-4 score gap if available — this is the key ultrafeedback advantage
    chosen_score   = sample.get("chosen_rating", None)
    rejected_score = sample.get("rejected_rating", None)
    if chosen_score is not None and rejected_score is not None:
        if (chosen_score - rejected_score) < 1.0:
            return False

    return True

print("\nFiltering for clean samples...")
ds_clean = ds.filter(is_clean, num_proc=2)
print(f"After filter: {len(ds_clean)} samples")


Filtering for clean samples...


Filter (num_proc=2):   0%|          | 0/61135 [00:00<?, ? examples/s]

After filter: 42044 samples


## Shuffle and allocate

In [7]:
ds_clean = ds_clean.shuffle(seed=SEED)

total_needed = SFT_SAMPLES + RM_SAMPLES + PPO_SAMPLES + TEST_SAMPLES
print(f"\nTotal needed: {total_needed}")
assert len(ds_clean) >= total_needed, \
    f"Not enough clean samples: {len(ds_clean)} < {total_needed}. Lower RM_SAMPLES or reduce filter strictness."

# Non-overlapping slices — each sample goes to exactly one split
sft_raw  = ds_clean.select(range(SFT_SAMPLES))
rm_raw   = ds_clean.select(range(SFT_SAMPLES,
                                  SFT_SAMPLES + RM_SAMPLES))
ppo_raw  = ds_clean.select(range(SFT_SAMPLES + RM_SAMPLES,
                                  SFT_SAMPLES + RM_SAMPLES + PPO_SAMPLES))
test_raw = ds_clean.select(range(SFT_SAMPLES + RM_SAMPLES + PPO_SAMPLES,
                                  SFT_SAMPLES + RM_SAMPLES + PPO_SAMPLES + TEST_SAMPLES))

print(f"\nAllocated — SFT: {len(sft_raw)} | RM: {len(rm_raw)} | PPO: {len(ppo_raw)} | Test: {len(test_raw)}")


Total needed: 16000

Allocated — SFT: 6000 | RM: 8500 | PPO: 1000 | Test: 500


## 5- format: SFT
{"text": full chat-templated string}
SFTTrainer expects either "text" column or "prompt"+"response" columns
 Using "text" with full chat template is cleaner for Qwen

In [13]:
def format_sft(sample):
    prompt   = get_prompt(sample)
    response = get_response(sample["chosen"])
    if not prompt or not response:
        return None

    messages = [
        {"role": "user",      "content": prompt},
        {"role": "assistant", "content": response}
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    if len(tokenizer(text, truncation=False)["input_ids"]) > MAX_LENGTH:
        return None

    return {"text": text, "prompt": prompt, "response": response}

print("\nFormatting SFT...")
sft_data = [r for s in sft_raw if (r := format_sft(s))]
print(f"SFT final: {len(sft_data)}")


Formatting SFT...
SFT final: 4242


## 6- Format: Reward Model
{"prompt": ..., "chosen": full_chosen_text, "rejected": full_rejected_text}
RewardTrainer expects this exact structure

In [14]:
def format_rm(sample):
    prompt        = get_prompt(sample)
    chosen_resp   = get_response(sample["chosen"])
    rejected_resp = get_response(sample["rejected"])
    if not prompt or not chosen_resp or not rejected_resp:
        return None

    chosen_text = tokenizer.apply_chat_template(
        [{"role": "user",      "content": prompt},
         {"role": "assistant", "content": chosen_resp}],
        tokenize=False, add_generation_prompt=False
    )
    rejected_text = tokenizer.apply_chat_template(
        [{"role": "user",      "content": prompt},
         {"role": "assistant", "content": rejected_resp}],
        tokenize=False, add_generation_prompt=False
    )
    if len(tokenizer(chosen_text,   truncation=False)["input_ids"]) > MAX_LENGTH:
        return None
    if len(tokenizer(rejected_text, truncation=False)["input_ids"]) > MAX_LENGTH:
        return None

    return {"prompt": prompt, "chosen": chosen_text, "rejected": rejected_text}
print("Formatting RM...")
rm_data = [r for s in rm_raw if (r := format_rm(s))]
print(f"RM final: {len(rm_data)}")

Formatting RM...
RM final: 5320


## 7- Format: PPO Prompts
Just the prompt — no response. PPO generates its own.
Also include the chat-template formatted prompt so PPO trainer 
can feed it directly to the model for generation.

In [15]:
def format_ppo(sample):
    prompt = get_prompt(sample)
    if not prompt:
        return None

    query = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True
    )
    return {"prompt": prompt, "query": query}

print("Formatting PPO prompts...")
ppo_data = [r for s in ppo_raw if (r := format_ppo(s))]
print(f"PPO final: {len(ppo_data)}")

Formatting PPO prompts...
PPO final: 1000


## 8- Format: Test

In [16]:
print("Formatting test set...")
test_data = [r for s in test_raw if (r := format_rm(s))]   
print(f"Test final: {len(test_data)}")

Formatting test set...
Test final: 303


## 9 - Save

In [17]:
def save_jsonl(data, path):
    with open(path, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")
    print(f"  Saved → {path}  ({len(data)} samples)")

print("\nSaving files...")
save_jsonl(sft_data,  OUTPUT_DIR / "sft_train.jsonl")
save_jsonl(rm_data,   OUTPUT_DIR / "reward_train.jsonl")
save_jsonl(ppo_data,  OUTPUT_DIR / "ppo_prompts.jsonl")
save_jsonl(test_data, OUTPUT_DIR / "test.jsonl")


Saving files...
  Saved → data/sft_train.jsonl  (4242 samples)
  Saved → data/reward_train.jsonl  (5320 samples)
  Saved → data/ppo_prompts.jsonl  (1000 samples)
  Saved → data/test.jsonl  (303 samples)


## Sanity Test

In [18]:
print("\n── Sanity Check ──")
s = rm_data[0]
print(f"Prompt:   {s['prompt'][:200]}")
print(f"Chosen:   {s['chosen'][:200]}")
print(f"Rejected: {s['rejected'][:200]}")
assert s["chosen"] != s["rejected"], "chosen == rejected, something is wrong"
print("\nchosen != rejected ✓")

print(f"""
── Final Split Summary ──
  sft_train.jsonl    {len(sft_data):>5} samples   (prompt + chosen)
  reward_train.jsonl {len(rm_data):>5} samples   (prompt + chosen + rejected)
  ppo_prompts.jsonl  {len(ppo_data):>5} samples   (prompt only)
  test.jsonl         {len(test_data):>5} samples   (prompt + chosen + rejected)
""")


── Sanity Check ──
Prompt:   During the World War II, where was the Canadian exile government located?
Chosen:   <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
During the World War II, where was the Canadian exile government located?<|im_end|>
<
Rejected: <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
During the World War II, where was the Canadian exile government located?<|im_end|>
<

chosen != rejected ✓

── Final Split Summary ──
  sft_train.jsonl     4242 samples   (prompt + chosen)
  reward_train.jsonl  5320 samples   (prompt + chosen + rejected)
  ppo_prompts.jsonl   1000 samples   (prompt only)
  test.jsonl           303 samples   (prompt + chosen + rejected)



In [19]:
import pandas as pd

df = pd.read_json("/kaggle/working/data/sft_train.jsonl", lines=True)
df.head()

,text,prompt,response
0,"<|im_start|>system\nYou are Qwen, created by A...","Make a meme title with the name ""Greta Thunber...","""Greta Thunberg Discovers the Most Climate-Des..."
1,"<|im_start|>system\nYou are Qwen, created by A...",Determine the type of quadrilateral formed by ...,Hello! I'd be happy to help you determine the ...
2,"<|im_start|>system\nYou are Qwen, created by A...",You are given a sentence in Spanish. Your job ...,بچه‌ها، بر اساس قوانین، شما معلمان بزرگ افسانه...
3,"<|im_start|>system\nYou are Qwen, created by A...",what are the minumum requirements for a ubuntu...,The minimum requirements for an Ubuntu ELK sta...
4,"<|im_start|>system\nYou are Qwen, created by A...","Given a sentence in the Japanese, provide an e...",ความเจ็บป่วยอาจจะส่งผลต่อโอกาสของเธอ แต่ในสัมภ...
